In [1]:

import asyncio
import warnings
import pandas as pd
from pathlib import Path

# Парсеры
from antimony import antimony_parser
from westmetall import westmetall_async
from lme import lme_selenium_async
from lbma import lbma_prescious_async

from cbr import cb_currency, cb_metalls
from nbk import nbk_tenge_async
from shmet import shmet_optimized_async
from kitco import kitco_parser_async

# Сервисные функции из service_layer
from service_layer import (
    read_db,
    save_db,
    check_df,
    check_and_save_pair,
    show_db,
    excel_to_csv_db
)

warnings.filterwarnings("ignore")


# ================================================================
# Пути к базам (реальная структура проекта)
# ================================================================
LME_PATH = Path("lme/data/LME_db_new.xlsx")
WESTMETALL_PATH = Path("westmetall/data/LME_westmetall_db.xlsx")

KITCO_PATH = Path("kitco/data/kitko_db.xlsx")
LBMA_PATH = Path("lbma/data/lbma_kitco_subs.xlsx")

ANTIMONY_PATH = Path("antimony/data/antimony.xlsx")

CB_CURRENCY_PATH = Path("cbr/data/cb_currency.xlsx")
CB_METALLS_PATH = Path("cbr/data/cb_metalls.xlsx")

NBK_PATH = Path("nbk/data/nbk_tenge.xlsx")
SHMET_PATH = Path("shmet/data/shmet_historical.xlsx")


# ================================================================
# Проверка целостности парных баз
# ================================================================
def db_check():
    """
    Проверка целостности парных баз с очисткой дубликатов
    и сохранением первого (более раннего) вхождения.
    """
    print("Проверка LME / Westmetall...")
    check_and_save_pair(
        LME_PATH,
        WESTMETALL_PATH,
        pair_name="LME / Westmetall",
        index=False,
    )

    print("Проверка Kitco / LBMA...")
    check_and_save_pair(
        KITCO_PATH,
        LBMA_PATH,
        pair_name="Kitco / LBMA",
        index=False,
    )
    
async def main():
    print("Parsing started...")

    tasks = {
        "lme": lme_selenium_async(),
        "antimony": antimony_parser(),
        "westmetall": westmetall_async(),
        "lbma": lbma_prescious_async(),
        "kitco": kitco_parser_async(),
        "cb_currency": cb_currency(),
        "cb_metalls": cb_metalls(),
        "nbk": nbk_tenge_async(),
        "shmet": shmet_optimized_async(),
    }

    # return_exceptions=True — чтобы падение одного парсера
    # не останавливало остальные
    results = await asyncio.gather(
        *tasks.values(),
        return_exceptions=True,
    )

    for name, result in zip(tasks.keys(), results):
        if isinstance(result, Exception):
            print(f"❌ Ошибка в {name}: {result}")

    print("All tasks are done!")
    print("+" * 64)
    print("Checking DB...")

    db_check()

    print("DB check completed!")
    print("+" * 64)
    print("Visual control")
    print("+" * 64)
    
    print("Converting Excel to CSV...")
    excel_to_csv_db()
    print("CSV conversion completed!")

    # Базовые металлы
    show_db("lme_selenium_db", LME_PATH, sheet_name=0)
    show_db("westmetall_db", WESTMETALL_PATH, sheet_name=0)

    # Драгоценные металлы
    show_db("kitco_db", KITCO_PATH, sheet_name=0)
    show_db("lbma_precious_db", LBMA_PATH, sheet_name=0)

    # Антимоний
    show_db("antimony_db", ANTIMONY_PATH, sheet_name=0)

    # ЦБ РФ: валюты (каждая на своем листе)
    for currency in [
        "USD",
        "EUR",
        "British_Pound",
        "China_Yuan",
        "Japanese_Yen",
        "Swiss_Franc",
    ]:
        show_db(
            f"cb_currency ({currency})",
            CB_CURRENCY_PATH,
            sheet_name=currency,
        )

    # ЦБ РФ: металлы
    show_db("cb_metalls_db", CB_METALLS_PATH, sheet_name=0)

    # Казахстан и SHMET
    show_db("nbk_tenge_db", NBK_PATH, sheet_name=0)
    show_db("shmet_historical_db", SHMET_PATH, sheet_name=0, show_head=True)


# Для Jupyter используем await, а не asyncio.run()
await main()

Parsing started...
🚀 LME parsing started...
antimony parsing is DONE
NBK_tenge parsing is DONE! (1761 строк)
✅ LME_main is done!!!
CB_metalls parsing is DONE!
WESTMETALL is done!!!
USD is done!
EUR is done!
Australian_Dollar is done!
China_Yuan is done!
British_Pound is done!
Kazakhstan_Tenge is done!
Japanese_Yen is done!
Swiss_Franc is done!
CB_currency parsing is DONE!
Произошла ошибка KITCO: Не найдены блоки <div class='grid'> на странице Kitco
SHMET is done!!!
[LBMA/Selenium] Открываем страницу LBMA...
[LBMA/Selenium] Обрабатываем Gold...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 1
[LBMA/Selenium] Gold: получено 10 строк
[LBMA/Selenium] Обрабатываем Silver...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 1
[LBMA/Selenium] Silver: получено 10 строк
[LBMA/Selenium] Обрабатываем Platinum...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 2
[LBMA/Selenium] Platinum: получено 10 строк
[LBMA/Selenium] Обрабатываем Palladium...
[LBMA/Selenium] Сетевых ответов

,date,nickel,zink,tin,aluminium,copper,lead
1236,2026-11-23,15215,3630.5,41250,3201.5,12165.0,1788.0
1237,2026-11-24,15060,3665.0,40825,3120.5,12035.0,1786.0
1238,2026-11-25,15035,3520.5,38200,2849.0,11703.0,1792.0
1239,2026-11-26,15665,3646.0,38425,2980.0,11965.0,1844.0
1240,2026-11-27,15745,3677.0,40275,2983.0,12311.5,1840.0


westmetall_db


,date,aluminium,copper,lead,nickel,zink,tin
1236,2026-11-23,3201.5,12165.0,1788.0,15215,3630.5,41250
1237,2026-11-24,3120.5,12035.0,1786.0,15060,3665.0,40825
1238,2026-11-25,2849.0,11703.0,1792.0,15035,3520.5,38200
1239,2026-11-26,2980.0,11965.0,1844.0,15665,3646.0,38425
1240,2026-11-27,2983.0,12311.5,1840.0,15745,3677.0,40275


kitco_db


,Date,Gold,Silver,Platinum,Palladium
14897,2026-11-23,3667.55,42.515,1342.85,908.85
14898,2026-11-24,3623.50,42.470,1298.15,879.35
14899,2026-11-25,3466.35,39.400,1240.75,843.75
14900,2026-11-26,3578.90,41.540,1294.50,877.95
14901,2026-11-27,3634.50,44.040,1322.70,898.45


... (18 outputs left out)

nbk_tenge_db


,date,Числовое значение,ДОЛЛАР США
1756,2026-11-24,1,455.12
1757,2026-11-25,1,458.28
1758,2026-11-26,1,458.57
1759,2026-11-27,1,458.85
1760,2026-11-28,1,458.13


shmet_historical_db


,date,price,unit
0,2020-01-10,48605,Yuan/MT
1,2020-01-14,48990,Yuan/MT
2,2020-01-15,49060,Yuan/MT
3,2020-01-16,48950,Yuan/MT
4,2020-01-17,48930,Yuan/MT
